# Process and View Results From Multiple Longstrips

In [ ]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal
import networkx as nx

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

import naatos_oct_tools.panelui.multistrip as multistrip
import naatos_oct_tools.panelui.singlestrip as singlestrip

In [2]:
# # Magics to autoreload submodules when they are modified
# %load_ext autoreload
# %autoreload 2

In [3]:
#%% Test Record Excel
dftests = pd.read_excel(
    r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx',
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
0,NaN,2025-03-26,GHL_pyapp_20250326T1258,oven aging test day 1,strip 7,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025-03-26,GHL_pyapp_20250326T1403,oven aging test day 1,strip 10,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-03-26,GHL_pyapp_20250326T1416,oven aging test day 1,strip 14,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2025-03-27,GHL_pyapp_20250327T1428,oven aging test day 2,strip 7,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2025-03-27,GHL_pyapp_20250327T1438,oven aging test day 2,strip 10,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,64.0,2025-05-12,GHL_pyapp_20250512T1258,0.350-0.374mg/mm lrg bag blu,ES0331#33 0.373,0.373,Vacuum,9.0,3.5,2.39775,...,308.5,143.0,165.5,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.22,4.15,"done,5",Simon
70,65.0,2025-05-12,GHL_pyapp_20250512T1303,0.350-0.374mg/mm lrg bag blu,ES0414#1 0.353,0.353,Vacuum,9.0,3.5,2.39775,...,308.5,143.0,165.5,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.22,4.14,"done,5",Simon
71,66.0,2025-05-12,GHL_pyapp_20250512T1341,Valves ES 4/16/2024 1-sided no reflo,#1,NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,trimmed ends to fit
72,67.0,2025-05-12,GHL_pyapp_20250512T1350,Valves ES 4/16/2024 1-sided no reflo,#2,NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,trimmed ends to fit


In [4]:
#%% Filter The Tests To View
# dfmasks = [
#     (dftests['Test date']<'2025-03-28') & (dftests['Test date']>='2025-03-26'),
#     dftests['Test name'] != 'GHL_pyapp_20250326T1258'
# ]
# dfmasks = [
#     (dftests['Test date']<='2025-04-15') & (dftests['Test date']>='2025-04-14')
# ]
# dfmasks = [
#     (dftests['Test date']=='2025-05-05')
# ]
dfmasks = [
    (dftests['Test date']=='2025-05-08') | (dftests['Test date']=='2025-05-12'),
    ~dftests['Batch'].str.startswith('Valves')
]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
22,17.0,2025-05-08,GHL_pyapp_20250508T1112,0.325-0.35,ES0408#17 0.327,0.327,Vacuum,9.0,3.5,2.39775,...,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218086,4.143631,"done,5",NUM_FOVS: remove +1
23,18.0,2025-05-08,GHL_pyapp_20250508T1128,0.325-0.35,ES0409#8 0.333,0.333,Vacuum,9.0,3.5,2.39775,...,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218077,4.143459,"done,5",NUM_FOVS: remove +1
24,19.0,2025-05-08,GHL_pyapp_20250508T1134,0.325-0.35,ES0410#18 0.333,0.333,Vacuum,9.0,3.5,2.39775,...,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218080,4.143517,"done,5",NUM_FOVS: remove +1
25,20.0,2025-05-08,GHL_pyapp_20250508T1143,0.375-0.4,ES0401#21 0.391,0.391,Vacuum,9.0,3.5,2.39775,...,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218082,4.143558,"done,5",NUM_FOVS: remove +1
26,21.0,2025-05-08,GHL_pyapp_20250508T1151,0.375-0.4,ES0403#33 0.392,0.392,Vacuum,9.0,3.5,2.39775,...,308.0,143.0,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218082,4.143550,"done,5",NUM_FOVS: remove +1
27,22.0,2025-05-08,GHL_pyapp_20250508T1158,0.375-0.4,ES0410#24 0.380,0.380,Vacuum,9.0,3.5,2.39775,...,308.0,143.0,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218091,4.143722,"done,5",NUM_FOVS: remove +1
28,23.0,2025-05-08,GHL_pyapp_20250508T1539,0.325-0.349mg/mm lrg bag blu,ES0410#20 0.338,0.338,Vacuum,9.0,3.5,2.39775,...,309.0,143.3,165.7,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218090,4.143710,"done,5",Simon (my data fields to be populated automati...
29,24.0,2025-05-08,GHL_pyapp_20250508T1545,0.325-0.349mg/mm lrg bag blu,ES0408#3 0.334,0.334,Vacuum,9.0,3.5,2.39775,...,309.0,143.3,165.7,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218089,4.143700,"done,5",Simon
30,25.0,2025-05-08,GHL_pyapp_20250508T1551,0.325-0.349mg/mm lrg bag blu,ES0408#13 0.341,0.341,Vacuum,9.0,3.5,2.39775,...,309.0,143.3,165.7,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218086,4.143628,"done,5",Simon
31,26.0,2025-05-08,GHL_pyapp_20250508T1555,0.325-0.349mg/mm lrg bag blu,ES0408#30 0.349,0.349,Vacuum,9.0,3.5,2.39775,...,309.0,143.3,165.7,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218091,4.143720,"done,5",Simon


In [5]:
#%% Load OCT study information
octstudies = [];
#folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

folder_temp = Path(r'D:\TEMP')

# Load data

In [6]:
#df = dfsteps[[x for x in dfsteps.columns.tolist() if x not in cols_ignore]]
dfloaded = pd.read_hdf(folder_temp/'oct_results_summary.hdf',key='table');

In [7]:
# strip down columns we will not use below
#df = df[[x for x in dfsteps.columns.tolist() if x not in cols_ignore]];
# join info from spreadsheet
dfloaded = pd.merge(dfloaded.reset_index(),dffilt,left_on='level_0',right_on='Test name');
dfloaded

,level_0,slice,pixel_depth_strip_top_sum_threshold,pixel_depth_strip_top,pixel_depth_strip_bot,pixel_depth_wax_center,px_wax_transverse_edges,wax_top_seed_candidate_px,wax_roi_valve_bottom,final_wax_bot_subset_px,...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
0,GHL_pyapp_20250508T1112,5,57.760000,181,247,222,"(56.33712121212121, 105.00769230769231)","(190, 80)",252,167,...,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218086,4.143631,"done,5",NUM_FOVS: remove +1
1,GHL_pyapp_20250508T1112,10,56.800000,182,246,223,"(56.755319148936174, 105.3936170212766)","(195, 80)",233,168,...,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218086,4.143631,"done,5",NUM_FOVS: remove +1
2,GHL_pyapp_20250508T1112,15,59.520000,183,244,226,"(56.82631578947368, 103.9396551724138)","(202, 79)",229,168,...,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218086,4.143631,"done,5",NUM_FOVS: remove +1
3,GHL_pyapp_20250508T1112,20,56.600000,184,244,231,"(55.68604651162791, 102.38636363636364)","(203, 78)",215,169,...,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218086,4.143631,"done,5",NUM_FOVS: remove +1
4,GHL_pyapp_20250508T1112,25,58.080000,184,243,223,"(53.572463768115945, 102.2843137254902)","(206, 77)",169,177,...,307.5,142.5,165.0,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.218086,4.143631,"done,5",NUM_FOVS: remove +1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80384,GHL_pyapp_20250512T1303,8240,51.040000,270,331,303,"(45.228873239436616, 121.35)","(288, 83)",186,186,...,308.5,143.0,165.5,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.220000,4.140000,"done,5",Simon
80385,GHL_pyapp_20250512T1303,8245,51.560000,270,330,303,"(44.877272727272725, 118.78289473684211)","(285, 81)",190,190,...,308.5,143.0,165.5,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.220000,4.140000,"done,5",Simon
80386,GHL_pyapp_20250512T1303,8250,55.440000,270,328,324,"(44.430851063829785, 116.775)","(283, 80)",189,189,...,308.5,143.0,165.5,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.220000,4.140000,"done,5",Simon
80387,GHL_pyapp_20250512T1303,8255,54.120000,268,326,305,"(44.61702127659574, 112.73170731707317)","(284, 78)",191,191,...,308.5,143.0,165.5,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.220000,4.140000,"done,5",Simon


In [8]:
dffilt.columns

Index(['Test ID', 'Test date', 'Test name', 'Batch', 'Strip', 'Wax (mg/mm)',
       'Fixture', 'X, FOV', 'Y, FOV', 'Z, FOV', 'X, pixel size',
       'Y, pixel size', 'Z, pixel size', 'Angle', 'Right edge (mm)',
       'Left edge (mm)', 'Strip length (mm)', 'Speed/Sensitivity',
       'Averaging (A-scan)', 'Refractive Index', 'File size (MB)',
       'Total size (GB)', 'ProcessingNotes', 'Notes'],
      dtype='object')

In [9]:
dffilt['Batch'].unique()

array(['0.325-0.35', '0.375-0.4', '0.325-0.349mg/mm lrg bag blu',
       '0.350-0.374mg/mm lrg bag blu'], dtype=object)

# Test Panel

In [10]:
pn.extension("tabulator")
import param

In [11]:
import naatos_oct_tools.panelui.metricdefs as metricdefs
metrics = metricdefs.metrics;


In [12]:
metrics


{'wax_top': ('distances', 'pixels'),
 'wax_thickness1': ('distances', 'pixels'),
 'wax_thickness2': ('distances', 'pixels'),
 'wax_width_px': ('distances', 'pixels'),
 'area': ('areas', 'pixels^2'),
 'area_filled': ('areas', 'pixels^2'),
 'area_convex': ('areas', 'pixels^2'),
 'seg_area_holes': ('areas', 'pixels^2'),
 'seg_area_to_areafilled': ('ratios', 'ratio'),
 'seg_area_to_areaconvex': ('ratios', 'ratio'),
 'seg_areafilled_to_areaconvex': ('ratios', 'ratio'),
 'num_paths': ('counts', 'counts')}

In [ ]:
OCTMultiStripMetricViewer = multistrip.OCTMultiStripMetricViewer;
OCTSingleStripMetricViewer = singlestrip.OCTSingleStripMetricViewer;

#import naatos_oct_tools.panelui.metricdefs as metricdefs
#metricdefs.m

In [14]:
obj2 = OCTMultiStripMetricViewer(
    ['GHL_pyapp_20250512T0947', 'GHL_pyapp_20250512T0951', 'GHL_pyapp_20250512T0955', 'GHL_pyapp_20250512T0959', 'GHL_pyapp_20250512T1002', 'GHL_pyapp_20250512T1117', 'GHL_pyapp_20250512T1121', 'GHL_pyapp_20250512T1125', 'GHL_pyapp_20250512T1130', 'GHL_pyapp_20250512T1133', 'GHL_pyapp_20250512T1137', 'GHL_pyapp_20250512T1140', 'GHL_pyapp_20250512T1144', 'GHL_pyapp_20250512T1149', 'GHL_pyapp_20250512T1152', 'GHL_pyapp_20250512T1158', 'GHL_pyapp_20250512T1201', 'GHL_pyapp_20250512T1204', 'GHL_pyapp_20250512T1208', 'GHL_pyapp_20250512T1211', 'GHL_pyapp_20250512T1215', 'GHL_pyapp_20250512T1218', 'GHL_pyapp_20250512T1222', 'GHL_pyapp_20250512T1225', 'GHL_pyapp_20250512T1230', 'GHL_pyapp_20250512T1238', 'GHL_pyapp_20250512T1241', 'GHL_pyapp_20250512T1245', 'GHL_pyapp_20250512T1248', 'GHL_pyapp_20250512T1251', 'GHL_pyapp_20250512T1254', 'GHL_pyapp_20250512T1258', 'GHL_pyapp_20250512T1303'],
    dfloaded
);

Loaded with 33 tests: ['GHL_pyapp_20250512T0947', 'GHL_pyapp_20250512T0951', 'GHL_pyapp_20250512T0955', 'GHL_pyapp_20250512T0959', 'GHL_pyapp_20250512T1002', 'GHL_pyapp_20250512T1117', 'GHL_pyapp_20250512T1121', 'GHL_pyapp_20250512T1125', 'GHL_pyapp_20250512T1130', 'GHL_pyapp_20250512T1133', 'GHL_pyapp_20250512T1137', 'GHL_pyapp_20250512T1140', 'GHL_pyapp_20250512T1144', 'GHL_pyapp_20250512T1149', 'GHL_pyapp_20250512T1152', 'GHL_pyapp_20250512T1158', 'GHL_pyapp_20250512T1201', 'GHL_pyapp_20250512T1204', 'GHL_pyapp_20250512T1208', 'GHL_pyapp_20250512T1211', 'GHL_pyapp_20250512T1215', 'GHL_pyapp_20250512T1218', 'GHL_pyapp_20250512T1222', 'GHL_pyapp_20250512T1225', 'GHL_pyapp_20250512T1230', 'GHL_pyapp_20250512T1238', 'GHL_pyapp_20250512T1241', 'GHL_pyapp_20250512T1245', 'GHL_pyapp_20250512T1248', 'GHL_pyapp_20250512T1251', 'GHL_pyapp_20250512T1254', 'GHL_pyapp_20250512T1258', 'GHL_pyapp_20250512T1303']


In [15]:
#fig = obj2._mkfigure('scatterVsLength',metrics);
#fig = fig[0].object
#fig.show(renderer='browser')

In [16]:
obj3 = OCTSingleStripMetricViewer(
    'GHL_pyapp_20250512T0947'
);

STUDY: GHL_pyapp_20250512T0947
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 0,
    'study_num_vtk_files': 0}


In [17]:
fig = obj3._mkfig()

Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE


In [ ]:
#fig.show(renderer='browser')

In [ ]:
#fig.data[0].name

In [ ]:
class OCTStripExplorer(pn.viewable.Viewer):
    # follow https://panel.holoviz.org/tutorials/intermediate/interactivity.html
    # from   the "with pn.rx" class

    data_table = param.DataFrame(doc="List Of Strips From Excel File")
    page_size = param.Integer(default=10, doc="Number of rows per page.", bounds=(1, None))

    # c_def_cols = ["unit","expname","run","TimeBeg","status","text"];
    # columns = param.ListSelector(
    #     default=["p_name", "t_state", "t_county", "p_year", "t_manu", "p_cap"]
    # )
    
    filtered_data = param.Parameter()

    number_of_rows = param.Parameter()

    # button1 = param.Action(label='Reset');
    # button2 = param.Action(label='Update');
    # button1 = param.Action(label='Refine');
    # button2 = param.Action(label='Reset');
    actionbutton = param.Action(lambda x: x.param.trigger('actionbutton'), label='Update plot!');
    refinebutton = param.Action(lambda x: x.param.trigger('refinebutton'), label='Refine choices');
    resetbutton = param.Action(lambda x: x.param.trigger('resetbutton'), label='Reset choices');
    # button2 = param.Action(lambda x: x.param.trigger('button2'), label='Start [spacebar]');
    # button3 = param.Action(lambda x: x.param.trigger('button3'), label='Insert[spacebar]');
    # button4 = param.Action(lambda x: x.param.trigger('button4'), label='Stop');
    wax_mass_range = param.Range(step=0.001);


    c_filters = {
        "Batch":param.ListSelector,
        "Test name":param.ListSelector,
    };

    multi_graph_obj = pn.pane.Str('multi graph origin');
    single_graph_obj = pn.pane.Str('single graph origin');

    @param.depends('refinebutton', watch=True)
    def _refine_button(self):
        self._update_or_add_params(filtered=True);
    
    def _update_or_add_params(self,filtered=False):
        df : pd.DataFrame;
        if(not filtered):
            df = self.data_table;
        else:
            df = self.filtered_data;

        for col,paramobj in self.c_filters.items():
            print('Working on',col,filtered)
            #print(self.data_table)
            # unique items

            items = df[col].unique().tolist();
    
            if(paramobj == param.ListSelector):
                if col not in self.param:
                    # add parameter
                    newParam = paramobj(default=sorted(items),objects=sorted(items));
                    print('Adding param',col,'df.shape',df.shape)
                    self.param.add_parameter(col,newParam);
                else:
                    # update parameter
                    #print('Updating param',col,'dffull.shape',self.data_table.shape)
                    print('Updating param',col,'df.shape',df.shape)
                    # reset available options
                    self.param[col].objects = items;
                    # reset selections (to all)
                    setattr(self,col,items);
            elif(paramobj == param.DateRange):
                print('DateRange (not implemented)')
                # if col not in self.param:
                #     # add parameter
                #     beg = sorted(self.data_table.value['TimeBeg'].unique())[0];
                #     end = sorted(self.data_table.value['TimeBeg'].unique(),reverse=True)[0];
                #     newParam = paramobj(default=(beg,end),bounds=(beg,end));
                #     print('Adding param',col,'dffull.shape',self.data_table.value.shape)
                #     obselfj.param.add_parameter(col,newParam);

        # Wax Valve Range Slider
        self.param.wax_mass_range.bounds = (df['Wax (mg/mm)'].min(),df['Wax (mg/mm)'].max())
        self.wax_mass_range = (df['Wax (mg/mm)'].min(),df['Wax (mg/mm)'].max())

    def _filter(self):
        dfrx = self.param.data_table.rx();

        # p_year_min = self.param.year.rx().rx.pipe(lambda x: x[0])
        # p_year_max = self.param.year.rx().rx.pipe(lambda x: x[1])
        # p_cap_min = self.param.capacity.rx().rx.pipe(lambda x: x[0])
        # p_cap_max = self.param.capacity.rx().rx.pipe(lambda x: x[1])

        # self.filtered_data = dfrx[
        #     dfrx.p_year.between(p_year_min, p_year_max)
        #     & dfrx.p_cap.between(p_cap_min, p_cap_max)
        # ][self.param.columns]
        
        # filter the columns
        # masks = [];
        # for col in self.c_filters.keys():

        #     pfilt = self.param[col].rx().rx;
        #     print(col,pfilt);
        #     masks.append(
        #         dfrx[col].isin(pfilt)
        #     )

        # # masks = [
        # #     dfrx
        # # ]
        # print(masks);
        # for col in self.c_filters.keys():
        #     dfrx = dfrx[dfrx.isin(self.param[col].rx())];

        #self.filtered_data = dfrx[dfrx['Batch'].isin(self.param['Batch'].rx())];
        # for col in self.c_filters.keys():
        #     print('filter col',col);
        #     dfrx = dfrx[dfrx.isin(self.param[col].rx())];
        # masks = [];
        # pfiltrxlist = [];
        # for col in self.c_filters.keys():
        #     print('filter col',col);
        #     pfiltrx = self.param[col].rx();
        #     pfiltrxlist.append(pfiltrx);
        #     masks.append( dfrx[col].isin(pfiltrx) );
        # #print(np.all(np.vstack(masks),axis=1))
        # dfrx = dfrx[ np.logical_and.reduce(masks,axis=0) ];
        print(self.wax_mass_range)
        p_wax_mass_range_min = self.param.wax_mass_range.rx().rx.pipe(lambda x: x[0]);
        p_wax_mass_range_max = self.param.wax_mass_range.rx().rx.pipe(lambda x: x[1]);
        dfrx = dfrx[
            dfrx['Batch'].isin(self.param['Batch'].rx()) &
            dfrx['Test name'].isin(self.param['Test name'].rx()) &
            dfrx['Wax (mg/mm)'].between( p_wax_mass_range_min , p_wax_mass_range_max )
        ];

        self.filtered_data = dfrx;

        self.number_of_rows = pn.rx("Scans: {len_df}").format(len_df=pn.rx(len)(dfrx))


    def __init__(self, data_table, **params):
        super().__init__(**params)
        self.data_table = data_table;
        #self.param.columns.objects = self.data_table.columns.to_list()
        self._update_or_add_params();
        self._reset_choices();
        self._filter();
    
        #self.wax_mass_range.param
        #self.param.wax_mass_range.bounds = (self.filtered_data.)

    @param.depends('resetbutton', watch=True)
    def _reset_choices(self):
        dfrx = self.param.data_table.rx();
        self.filtered_data = dfrx;
        self._update_or_add_params();
    
    @param.depends('refinebutton', watch=True)
    def _refine_choices(self):
        print('Refine choices');
        #self._update_or_add_params(filtered=True);

    def click_handling_2_multigraph(self,event):
        print("click_handling_2_multigraph datatype:{:s} data:{:s}", type(event), str(event));
    
        if not event and (self.multi_graph_obj.plotpane.object is None):
            print( "No point clicked" );
            return;
            pass;
        try:
            point = event["points"][0]
            curvenumber = point['curveNumber'];
            index = point['pointIndex']
            x = point['x']
            y = point['y']
            if(type(x) is not str):
                # x was a number
                # so.... find the name buried inside the plotly plot legend
                scanname = self.multi_graph_obj.plotpane.object.data[curvenumber].name;
            else:
                scanname = x;
        except Exception as ex:
            print( f"You clicked the Plotly Chart! I could not determine the point: {ex}" )
            return;
        
        print( f"**You clicked point index {index} at ({x}, {y}) on curve ({scanname}) in the Plotly Chart!**" )

        # instance a new single graph obj
        self.single_graph_obj = OCTSingleStripMetricViewer(scanname);
        self.tabs[2] = ('3-SinglestripMetrics',pn.Column(self.single_graph_obj));

    @param.depends('actionbutton', watch=True)
    def _update_plot(self):
        print('Update plot!');
        df = self.filtered_data.rx.value;
        #print(self.filtered_data['Test name'].unique().tolist())
        self.multi_graph_obj = OCTMultiStripMetricViewer(df['Test name'].unique().tolist(),dfloaded=dfloaded);
        print(self.multi_graph_obj.plotpane)
        iclickbind = pn.bind(self.click_handling_2_multigraph, self.multi_graph_obj.plotpane.param.click_data);
        self.tabs[1] = ('2-MultistripMetrics',pn.Column(iclickbind,self.multi_graph_obj));
        #print('Update plot! 2');

    def __panel__(self):
        stylesheet = """
        .tabulator-cell {
            font-size: 9px;
        }
        """
        main_panel = pn.Column(
            # pn.Row(
            #     pn.widgets.MultiChoice.from_param(self.param.columns, width=400),
            #     pn.Column(self.param.year, self.param.capacity),
            # ),
            pn.Row(
                pn.Column(
                    self.param.actionbutton,self.param.refinebutton,self.param.resetbutton
                ),
                *[self.param[x] for x in self.c_filters.keys()],
                self.param.wax_mass_range,
            ),
            self.number_of_rows,
            pn.Row(
                pn.widgets.Tabulator(self.filtered_data, page_size=10, pagination="remote",stylesheets=[stylesheet],sizing_mode= 'stretch_width',width=1200)
            ),
            #width_policy=''
        );
        self.tabs = pn.Tabs(
            ('1-Main',main_panel),
            ('2-MultistripMetrics',self.multi_graph_obj),
            ('3-SinglestripMetrics',self.single_graph_obj),
            
            #width=1000
        )
        return self.tabs;
    
obj = OCTStripExplorer(dffilt);
finpn = pn.Column(obj);


In [ ]:
obj.c_filters.keys()

In [ ]:
obj.param['Batch'].rx()

In [ ]:
# class EditableRange(pn.viewable.Viewer):
#     value = param.Range(doc="A numeric range.")
#     width = param.Integer(default=300)

#     def __init__(self, **params):
#         self._start_input = pn.widgets.FloatInput()
#         self._end_input = pn.widgets.FloatInput(align='end')
#         super().__init__(**params)
#         self._layout = pn.Row(self._start_input, self._end_input)
#         self._sync_widgets()

#     def __panel__(self):
#         return self._layout

#     @param.depends('value', 'width', watch=True)
#     def _sync_widgets(self):
#         self._start_input.name = self.name
#         self._start_input.value = self.value[0]
#         self._end_input.value = self.value[1]
#         self._start_input.width = self.width//2
#         self._end_input.width = self.width//2

#     @param.depends('_start_input.value', '_end_input.value', watch=True)
#     def _sync_params(self):
#         self.value = (self._start_input.value, self._end_input.value)

# # range_widget = EditableRange(name='Range', value=(0, 10))

# # finpn = pn.Column(
# #     '#### This is a custom widget',
# #     range_widget
# # );


In [ ]:
# def session_key_func(request):
#     key = request.arguments.get('expname', [def_expname.encode()])[0]+'_'+request.arguments.get('unit', [def_unit.encode()])[0]+'_'+request.arguments.get('run', [def_run.encode()])[0];
#     print('session_key_func',key);
#     return key;

# #pn.extension(template='material', session_key_func=session_key_func)
# pn.extension(session_key_func=session_key_func)

# obj = DataExplorer(data=dffull)
# obj2 = Plotter1App(dffull = dfraw, dfbuilt = dfbuilt);

# def page1():
#     #return obj.servable(location=True)
#     return obj;

# def page_rundetail():
#     #pn.state.location.sync(obj2,['unit']);
#     pn.state.location.sync(obj2, ['expname','unit','run']);
#     #return obj2.servable()
#     return obj2;

def page1():
    #return finpn;
    return OCTStripExplorer(dffilt);

ROUTES = {
    "": page1,
    #"rundetail": page_rundetail
}
# try:
#     serve.stop();
# except Exception as e:
#     print(e)

#serve = pn.serve(ROUTES, port=5006,location=True,verbose=True,admin=True);
page1().servable(target='page1');

In [ ]:
# try:
#     print('Check existing server',str(server))
#     print('Stopping server...')
#     server.stop();
#     del server;
# except:
#     print('Server was not running.')
#     pass;
# server = pn.serve(finpn,show=True)

In [ ]:
# server.stop()